Exntinção 

In [2]:
#!/usr/bin/env python3
"""
===============================================================================
Passo 0: Tabela Permanente de Coeficientes de Extinção (S-PLUS & J-PAS)
===============================================================================
LEGENDA DAS SEÇÕES:
- SEÇÃO 1: Definição dos coeficientes R_lambda = A_lambda / E(B-V)
- SEÇÃO 2: Estruturação das tabelas com Astropy
- SEÇÃO 3: Exportação em formato FITS
===============================================================================
"""

import os
from astropy.table import Table, Column

def generate_extinction_fits(output_path="extinction_coefficients_splus_jpas.fits"):
    print("--- [Passo 0] Gerando Tabela de Coeficientes de Extinção ---")
    
    # -------------------------------------------------------------------------
    # SEÇÃO 1: Definição dos Coeficientes R_lambda (R_V = 3.1, S&F 2011)
    # -------------------------------------------------------------------------
    # Coeficientes para os 12 filtros do S-PLUS
    splus_filters = {
        'u': 4.148, 'F378': 4.020, 'F395': 3.840, 'F410': 3.690, 
        'F430': 3.520, 'g': 3.237, 'F515': 2.820, 'r': 2.270, 
        'F660': 2.150, 'i': 1.682, 'F861': 1.250, 'z': 1.096
    }
    
    # Coeficientes para os 56 filtros do J-PAS (Amostragem do UV ao Infravermelho)
    jpas_filters = {
        'uJAVA': 4.150, 'J0378': 4.020, 'J0390': 3.900, 'J0400': 3.800, 'J0410': 3.690,
        'J0420': 3.590, 'J0430': 3.520, 'J0440': 3.420, 'J0450': 3.330, 'J0460': 3.250,
        'J0470': 3.180, 'J0480': 3.110, 'J0490': 3.040, 'J0500': 2.970, 'J0510': 2.880,
        'J0520': 2.790, 'J0530': 2.710, 'J0540': 2.640, 'J0550': 2.570, 'J0560': 2.510,
        'J0570': 2.450, 'J0580': 2.400, 'J0590': 2.350, 'J0600': 2.300, 'J0610': 2.260,
        'J0620': 2.220, 'J0630': 2.180, 'J0640': 2.140, 'J0650': 2.100, 'J0660': 2.070,
        'J0670': 2.030, 'J0680': 2.000, 'J0690': 1.960, 'J0700': 1.930, 'J0710': 1.890,
        'J0720': 1.860, 'J0730': 1.820, 'J0740': 1.790, 'J0750': 1.760, 'J0760': 1.730,
        'J0770': 1.700, 'J0780': 1.670, 'J0790': 1.640, 'J0800': 1.610, 'J0810': 1.580,
        'J0820': 1.550, 'J0830': 1.520, 'J0840': 1.490, 'J0850': 1.460, 'J0860': 1.430,
        'J0870': 1.400, 'J0880': 1.370, 'J0890': 1.340, 'J0900': 1.310, 'J0910': 1.280,
        'rSDSS': 2.270
    }

    # -------------------------------------------------------------------------
    # SEÇÃO 2: Construção da Tabela Astropy
    # -------------------------------------------------------------------------
    surveys = []
    filters = []
    r_values = []

    for f_name, r_val in splus_filters.items():
        surveys.append("S-PLUS")
        filters.append(f_name)
        r_values.append(r_val)

    for f_name, r_val in jpas_filters.items():
        surveys.append("J-PAS")
        filters.append(f_name)
        r_values.append(r_val)

    table = Table()
    table['survey'] = Column(surveys, description="Nome do Survey (S-PLUS ou J-PAS)")
    table['filter'] = Column(filters, description="Nome da banda/filtro")
    table['R_lambda'] = Column(r_values, unit="mag/mag", description="Razao A_lambda / E(B-V)")

    # -------------------------------------------------------------------------
    # SEÇÃO 3: Salvando em formato FITS
    # -------------------------------------------------------------------------
    table.write(output_path, format='fits', overwrite=True)
    print(f"[Sucesso] Arquivo FITS gerado com {len(table)} filtros salvos em: {output_path}")

if __name__ == "__main__":
    generate_extinction_fits()

--- [Passo 0] Gerando Tabela de Coeficientes de Extinção ---
[Sucesso] Arquivo FITS gerado com 68 filtros salvos em: extinction_coefficients_splus_jpas.fits


# 1. Gerar a tabela de coeficientes de extinção FITS (apenas uma vez)


# 2. Testar o leitor de configuração


# 3. Executar o carregamento e filtragem das amostras APOGEE + Gaia


In [3]:
%%writefile config.yaml
# ==============================================================================
# Configuração Central do Pipeline Customizado
# ==============================================================================

paths:
  spectroscopy_fits: "/home/murilo/data/spectroscopy/Apogee_Dr17.fits"
  gaia_auxiliary_fits: "/home/murilo/data/gaia/source_id_catalog_evb_gaia.fits"
  photometry_dir: "/run/user/1000/gvfs/smb-share:server=150.163.28.39,share=storage-share/Samples/X-PLUS/data"
  extinction_coefficients_fits: "./extinction_coefficients_splus_jpas.fits"
  output_dir: "./output_fits"

apogee_cuts:
  snr_min: 10.0
  teff_err_max: 300.0
  logg_err_max: 0.40
  feh_err_max: 0.40
  feh_min: -4.5
  feh_max: 1.5

gaia_cuts:
  ruwe_max: 1.4
  parallax_over_error_min: 5.0

photometry:
  max_mag_error: 0.10
  survey: "S-PLUS"

Overwriting config.yaml


In [4]:
%%writefile config_loader.py
import os
import yaml

def load_config(config_path="config.yaml"):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Arquivo de configuração não encontrado: {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
        
    os.makedirs(config['paths']['output_dir'], exist_ok=True)
    return config

Overwriting config_loader.py


In [ ]:
#Copiar o arquivo para o disco local via shutil

#Execute este código para realizar a cópia local direta sem passar pela serialização do Astropy:
#Python

#import os
#import shutil

#orig_path = "/run/user/1000/gvfs/smb-share:server=150.163.28.39,share=storage-share/Samples/Spectroscopic_Surveys/Apogee_Dr17.fits"
#local_path = "/home/murilo/data/spectroscopy/Apogee_Dr17.fits"

#os.makedirs(os.path.dirname(local_path), exist_ok=True)

#print(f"Copiando arquivo original da rede para {local_path}...")
#shutil.copyfile(orig_path, local_path)

#final_size_mb = os.path.getsize(local_path) / (1024 * 1024)
#print(f"=== SUCESSO! Cópia local criada com {final_size_mb:.2f} MB ===")

Copiando arquivo original da rede para /home/murilo/data/spectroscopy/Apogee_Dr17.fits...
=== SUCESSO! Cópia local criada com 491.39 MB ===


In [5]:
import os
import warnings
import numpy as np
import pandas as pd
from astropy.table import Table, join
from astropy.io.fits.verify import VerifyWarning
from config_loader import load_config

# Silencia avisos cosméticos de formatação FITS do Astropy
warnings.simplefilter('ignore', category=VerifyWarning)
warnings.filterwarnings('ignore', message='.*did not parse as fits unit.*')

print("=== PASSO 2: Carregamento, Extração do ID Gaia e Filtragem ===")

# -----------------------------------------------------------------------------
# 1. Carregar Configurações do Projeto
# -----------------------------------------------------------------------------
cfg = load_config()
out_dir = cfg['paths']['output_dir']

# -----------------------------------------------------------------------------
# 2. Leitura e Extração do ID Gaia a partir da URL do VizieR
# -----------------------------------------------------------------------------
apogee_path = cfg['paths']['spectroscopy_fits']
print(f"[1/4] Lendo catálogo local APOGEE: {apogee_path}")
apogee_tbl = Table.read(apogee_path, format='fits')

raw_gaia = apogee_tbl['GaiaEDR3']
if hasattr(raw_gaia, 'filled'):
    raw_gaia = raw_gaia.filled('')

# Converte bytes/Unicode para Série do Pandas
sid_series = pd.Series(np.array(raw_gaia).astype('U'))

print(" -> Extraindo o ID do Gaia da URL do VizieR...")
# Extrai sequências numéricas longas (IDs do Gaia têm entre 15 e 20 dígitos)
extracted_ids = sid_series.str.extract(r'(\d{15,20})', expand=False)

# Atribui a coluna source_id convertida para int64
apogee_tbl['source_id'] = pd.to_numeric(extracted_ids, errors='coerce').fillna(0).astype(np.int64)

valid_sid = apogee_tbl['source_id'] > 0
print(f" -> Estrelas com source_id EXTRAÍDO e VÁLIDO: {valid_sid.sum()} de {len(apogee_tbl)}")

# -----------------------------------------------------------------------------
# 3. Aplicação dos Cortes de Qualidade Espectroscópica
# -----------------------------------------------------------------------------
cuts = cfg['apogee_cuts']
mask_quality = (
    (apogee_tbl['SNR'] >= cuts['snr_min']) &
    (apogee_tbl['e_Teff'] <= cuts['teff_err_max']) &
    (apogee_tbl['e_logg'] <= cuts['logg_err_max']) &
    (apogee_tbl['e_[Fe/H]'] <= cuts['feh_err_max']) &
    (apogee_tbl['[Fe/H]'] >= cuts['feh_min']) &
    (apogee_tbl['[Fe/H]'] <= cuts['feh_max'])
)

apogee_clean = apogee_tbl[mask_quality & valid_sid]

print(f" -> APOGEE original: {len(apogee_tbl)} estrelas")
print(f" -> APOGEE filtrado (Qualidade + source_id válido): {len(apogee_clean)} estrelas")

# -----------------------------------------------------------------------------
# 4. Leitura e Cruzamento com Catálogo Auxiliar Gaia Local
# -----------------------------------------------------------------------------
gaia_path = cfg['paths']['gaia_auxiliary_fits']
print(f"[2/4] Lendo catálogo auxiliar Gaia local: {gaia_path}")
gaia_tbl = Table.read(gaia_path, format='fits', memmap=True)

if gaia_tbl['source_id'].dtype != np.int64:
    gaia_tbl['source_id'] = gaia_tbl['source_id'].astype(np.int64)

print("[3/4] Cruzando amostras via 'source_id'...")
matched_tbl = join(apogee_clean, gaia_tbl, keys='source_id', join_type='inner')
print(f" -> Objetos em comum encontrados: {len(matched_tbl)}")

# -----------------------------------------------------------------------------
# 5. Filtros Astrométricos Gaia
# -----------------------------------------------------------------------------
g_cuts = cfg['gaia_cuts']
mask_gaia = (
    (matched_tbl['ruwe'] <= g_cuts['ruwe_max']) &
    (matched_tbl['parallax_over_error'] >= g_cuts['parallax_over_error_min'])
)

final_sample = matched_tbl[mask_gaia]
print(f" -> Amostra final (RUWE <= {g_cuts['ruwe_max']}, Plx SNR >= {g_cuts['parallax_over_error_min']}): {len(final_sample)} estrelas")

# -----------------------------------------------------------------------------
# 6. Salvar Amostra Filtrada em FITS
# -----------------------------------------------------------------------------
os.makedirs(out_dir, exist_ok=True)
output_fits_path = os.path.join(out_dir, "apogee_gaia_sample_filtered.fits")
print(f"[4/4] Salvando amostra filtrada em: {output_fits_path}")

final_sample.write(output_fits_path, format='fits', overwrite=True)
print("=== PASSO 2 CONCLUÍDO COM SUCESSO! ===")

=== PASSO 2: Carregamento, Extração do ID Gaia e Filtragem ===
[1/4] Lendo catálogo local APOGEE: /home/murilo/data/spectroscopy/Apogee_Dr17.fits
 -> Extraindo o ID do Gaia da URL do VizieR...
 -> Estrelas com source_id EXTRAÍDO e VÁLIDO: 728274 de 733901
 -> APOGEE original: 733901 estrelas
 -> APOGEE filtrado (Qualidade + source_id válido): 640027 estrelas
[2/4] Lendo catálogo auxiliar Gaia local: /home/murilo/data/gaia/source_id_catalog_evb_gaia.fits
[3/4] Cruzando amostras via 'source_id'...


 -> Objetos em comum encontrados: 195566
 -> Amostra final (RUWE <= 1.4, Plx SNR >= 5.0): 151408 estrelas
[4/4] Salvando amostra filtrada em: ./output_fits/apogee_gaia_sample_filtered.fits
=== PASSO 2 CONCLUÍDO COM SUCESSO! ===


In [ ]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
from astropy.table import Table, join, vstack
from astropy.io.fits.verify import VerifyWarning
from config_loader import load_config

warnings.simplefilter('ignore', category=VerifyWarning)
warnings.filterwarnings('ignore', message='.*did not parse as fits unit.*')

print("=== PASSO 3: Cópia Local (NVMe) e Cruzamento Fotométrico ===")

# -----------------------------------------------------------------------------
# 1. Carregar Configurações e Amostra do Passo 2
# -----------------------------------------------------------------------------
cfg = load_config()
out_dir = cfg['paths']['output_dir']
local_cache_dir = os.path.join(out_dir, "photometry_cache")
os.makedirs(local_cache_dir, exist_ok=True)

input_sample_path = os.path.join(out_dir, "apogee_gaia_sample_filtered.fits")
print(f"[1/5] Lendo amostra filtrada (APOGEE + Gaia): {input_sample_path}")
base_tbl = Table.read(input_sample_path, format='fits')
print(f" -> Total de objetos de entrada: {len(base_tbl)}")

# -----------------------------------------------------------------------------
# 2. Localizar Arquivos de Fotometria na Rede (Busca Robusta)
# -----------------------------------------------------------------------------
photo_path = cfg['paths'].get('photometry_fits') or cfg['paths'].get('photometry_dir')

remote_files = []
if os.path.isfile(photo_path):
    remote_files = [photo_path]
elif os.path.isdir(photo_path):
    for root, _, files in os.walk(photo_path):
        for f in files:
            if f.lower().endswith(('.fits', '.fits.gz')):
                remote_files.append(os.path.join(root, f))

if not remote_files:
    raise FileNotFoundError(
        f"Nenhum arquivo .fits encontrado em: {photo_path}\n"
        f"Verifique se o caminho no config.yaml está correto ou se a pasta da rede está montada."
    )

print(f"[2/5] Encontrado(s) {len(remote_files)} arquivo(s) de fotometria na rede.")

# -----------------------------------------------------------------------------
# 3. Copiar Arquivos para o NVMe Local
# -----------------------------------------------------------------------------
local_files = []
for rf in remote_files:
    fname = os.path.basename(rf)
    lf = os.path.join(local_cache_dir, fname)
    
    # Copia apenas se o arquivo local não existir ou tiver tamanho diferente
    if not os.path.exists(lf) or os.path.getsize(lf) != os.path.getsize(rf):
        print(f" -> Copiando {fname} para o cache local...")
        shutil.copy2(rf, lf)
    local_files.append(lf)

# -----------------------------------------------------------------------------
# 4. Leitura do Disco Local (memmap=False) e Unificação
# -----------------------------------------------------------------------------
print("[3/5] Lendo e empilhando catálogos fotométricos...")
photo_tables = []
for f in local_files:
    t = Table.read(f, format='fits', memmap=False)
    photo_tables.append(t)

if len(photo_tables) == 1:
    photo_tbl = photo_tables[0]
else:
    print(f" -> Empilhando {len(photo_tables)} tabelas fotométricas...")
    photo_tbl = vstack(photo_tables, metadata_conflicts='silent')

# Trata a chave source_id na fotometria
if 'source_id' not in photo_tbl.colnames:
    possible_keys = ['GaiaEDR3', 'GAIAEDR3_SOURCE_ID', 'ID', 'source_id_gaia', 'Source']
    found_key = next((k for k in possible_keys if k in photo_tbl.colnames), None)
    if found_key:
        photo_tbl['source_id'] = photo_tbl[found_key]
    else:
        raise KeyError(f"Chave 'source_id' não encontrada no catálogo de fotometria. Colunas disponíveis: {photo_tbl.colnames}")

# Extração de números de ID caso venham em URLs ou bytes
raw_photo_sid = photo_tbl['source_id']
if hasattr(raw_photo_sid, 'filled'):
    raw_photo_sid = raw_photo_sid.filled('')

sid_series = pd.Series(np.array(raw_photo_sid).astype('U'))
extracted_ids = sid_series.str.extract(r'(\d{15,20})', expand=False).fillna(sid_series)
photo_tbl['source_id'] = pd.to_numeric(extracted_ids, errors='coerce').fillna(0).astype(np.int64)

# -----------------------------------------------------------------------------
# 5. Cruzamento com a Amostra Filtrada e Cortes Fotométricos
# -----------------------------------------------------------------------------
print("[4/5] Cruzando amostra via 'source_id'...")
matched_master = join(base_tbl, photo_tbl, keys='source_id', join_type='inner', table_names=['_APOGEE', '_SPLUS'])
print(f" -> Estrelas com fotometria pareada: {len(matched_master)}")

print("[5/5] Aplicando cortes de erro fotométrico...")
bands = ['u', 'J0378', 'J0395', 'J0410', 'J0430', 'g', 'J0515', 'r', 'J0660', 'i', 'J0861', 'z']
err_max = cfg.get('photometry_cuts', {}).get('mag_err_max', 0.10)

mask_phot = np.ones(len(matched_master), dtype=bool)

for b in bands:
    possible_err_cols = [f'e_{b}', f'e_{b}_auto', f'e_{b}_aper', f'{b}_err', f'e_{b.lower()}', f'e_{b.upper()}']
    err_col = next((c for c in possible_err_cols if c in matched_master.colnames), None)
    
    if err_col:
        err_vals = np.nan_to_num(matched_master[err_col], nan=99.0)
        mask_phot &= (err_vals > 0) & (err_vals <= err_max)

final_master = matched_master[mask_phot]
print(f" -> Amostra final limpa (erro <= {err_max} mag): {len(final_master)} estrelas")

# -----------------------------------------------------------------------------
# 6. Salvar Tabela Mestre
# -----------------------------------------------------------------------------
output_master_path = os.path.join(out_dir, "apogee_gaia_splus_master.fits")
print(f"Salvando amostra mestre final em: {output_master_path}")

final_master.write(output_master_path, format='fits', overwrite=True)
print("=== PASSO 3 CONCLUÍDO COM SUCESSO! ===")

=== PASSO 3: Cópia Local (NVMe) e Cruzamento Fotométrico ===
[1/5] Lendo amostra filtrada (APOGEE + Gaia): ./output_fits/apogee_gaia_sample_filtered.fits
 -> Total de objetos de entrada: 151408


FileNotFoundError: Nenhum arquivo .fits encontrado em: /run/user/1000/gvfs/smb-share:server=150.163.28.39,share=storage-share/Samples/X-PLUS/data
Verifique se o caminho no config.yaml está correto ou se a pasta da rede está montada.

In [16]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
from astropy.table import Table, join, vstack
from astropy.io.fits.verify import VerifyWarning
from config_loader import load_config

warnings.simplefilter('ignore', category=VerifyWarning)
warnings.filterwarnings('ignore', message='.*did not parse as fits unit.*')

print("=== PASSO 3: Cópia Local (NVMe) e Cruzamento Fotométrico ===")

# -----------------------------------------------------------------------------
# 1. Carregar Configurações e Amostra do Passo 2
# -----------------------------------------------------------------------------
cfg = load_config()
out_dir = cfg['paths']['output_dir']
local_cache_dir = os.path.join(out_dir, "photometry_cache")
os.makedirs(local_cache_dir, exist_ok=True)

input_sample_path = os.path.join(out_dir, "apogee_gaia_sample_filtered.fits")
print(f"[1/5] Lendo amostra filtrada (APOGEE + Gaia): {input_sample_path}")
base_tbl = Table.read(input_sample_path, format='fits')
print(f" -> Total de objetos de entrada: {len(base_tbl)}")

# -----------------------------------------------------------------------------
# 2. Localizar Arquivos de Fotometria na Rede (Compatível com GVFS/SMB)
# -----------------------------------------------------------------------------
photo_path = cfg['paths'].get('photometry_fits') or cfg['paths'].get('photometry_dir')
print(f" -> Caminho de fotometria configurado: {photo_path}")

if not os.path.exists(photo_path):
    raise FileNotFoundError(f"O caminho configurado não existe ou a rede caiu: {photo_path}")

remote_files = []

if os.path.isfile(photo_path):
    remote_files = [photo_path]
else:
    # Função recursiva baseada em os.listdir para evitar falhas do os.walk no GVFS
    def find_fits_gvfs(dir_path):
        found = []
        try:
            for item in os.listdir(dir_path):
                full_path = os.path.join(dir_path, item)
                if item.lower().endswith(('.fits', '.fits.gz')):
                    found.append(full_path)
                elif os.path.isdir(full_path):
                    found.extend(find_fits_gvfs(full_path))
        except Exception as e:
            print(f" ⚠️  Aviso ao listar {dir_path}: {e}")
        return found

    remote_files = find_fits_gvfs(photo_path)

if not remote_files:
    raise FileNotFoundError(
        f"Nenhum arquivo .fits encontrado em: {photo_path}\n"
        f"Conteúdo do diretório: {os.listdir(photo_path) if os.path.exists(photo_path) else 'Inacessível'}"
    )

print(f"[2/5] Encontrado(s) {len(remote_files)} arquivo(s) de fotometria na rede.")

# -----------------------------------------------------------------------------
# 3. Copiar Arquivos para o Cache Local (NVMe)
# -----------------------------------------------------------------------------
local_files = []
for rf in remote_files:
    fname = os.path.basename(rf)
    lf = os.path.join(local_cache_dir, fname)
    
    if not os.path.exists(lf) or os.path.getsize(lf) != os.path.getsize(rf):
        print(f" -> Copiando {fname} para NVMe local...")
        shutil.copy2(rf, lf)
    local_files.append(lf)

# -----------------------------------------------------------------------------
# 4. Leitura do Disco Local e Unificação
# -----------------------------------------------------------------------------
print("[3/5] Lendo e empilhando catálogos fotométricos...")
photo_tables = []
for f in local_files:
    t = Table.read(f, format='fits', memmap=False)
    photo_tables.append(t)

if len(photo_tables) == 1:
    photo_tbl = photo_tables[0]
else:
    print(f" -> Empilhando {len(photo_tables)} tabelas fotométricas...")
    photo_tbl = vstack(photo_tables, metadata_conflicts='silent')

# Trata a chave source_id na fotometria
if 'source_id' not in photo_tbl.colnames:
    possible_keys = ['GaiaEDR3', 'GAIAEDR3_SOURCE_ID', 'ID', 'source_id_gaia', 'Source', 'SOURCE_ID']
    found_key = next((k for k in possible_keys if k in photo_tbl.colnames), None)
    if found_key:
        photo_tbl['source_id'] = photo_tbl[found_key]
    else:
        raise KeyError(f"Chave 'source_id' não encontrada na fotometria. Colunas disponíveis: {photo_tbl.colnames}")

# Extração dos números de ID caso venham em URLs ou bytes
raw_photo_sid = photo_tbl['source_id']
if hasattr(raw_photo_sid, 'filled'):
    raw_photo_sid = raw_photo_sid.filled('')

sid_series = pd.Series(np.array(raw_photo_sid).astype('U'))
extracted_ids = sid_series.str.extract(r'(\d{15,20})', expand=False).fillna(sid_series)
photo_tbl['source_id'] = pd.to_numeric(extracted_ids, errors='coerce').fillna(0).astype(np.int64)

# -----------------------------------------------------------------------------
# 5. Cruzamento com Amostra Filtrada e Cortes
# -----------------------------------------------------------------------------
print("[4/5] Cruzando amostra via 'source_id'...")
matched_master = join(base_tbl, photo_tbl, keys='source_id', join_type='inner', table_names=['_APOGEE', '_SPLUS'])
print(f" -> Estrelas com fotometria pareada: {len(matched_master)}")

print("[5/5] Aplicando cortes de erro fotométrico...")
bands = ['u', 'J0378', 'J0395', 'J0410', 'J0430', 'g', 'J0515', 'r', 'J0660', 'i', 'J0861', 'z']
err_max = cfg.get('photometry_cuts', {}).get('mag_err_max', 0.10)

mask_phot = np.ones(len(matched_master), dtype=bool)

for b in bands:
    possible_err_cols = [f'e_{b}', f'e_{b}_auto', f'e_{b}_aper', f'{b}_err', f'e_{b.lower()}', f'e_{b.upper()}']
    err_col = next((c for c in possible_err_cols if c in matched_master.colnames), None)
    
    if err_col:
        err_vals = np.nan_to_num(matched_master[err_col], nan=99.0)
        mask_phot &= (err_vals > 0) & (err_vals <= err_max)

final_master = matched_master[mask_phot]
print(f" -> Amostra final limpa (erro <= {err_max} mag): {len(final_master)} estrelas")

# -----------------------------------------------------------------------------
# 6. Salvar Tabela Mestre Final
# -----------------------------------------------------------------------------
output_master_path = os.path.join(out_dir, "apogee_gaia_splus_master.fits")
print(f"Salvando amostra mestre final em: {output_master_path}")

final_master.write(output_master_path, format='fits', overwrite=True)
print("=== PASSO 3 CONCLUÍDO COM SUCESSO! ===")

=== PASSO 3: Cópia Local (NVMe) e Cruzamento Fotométrico ===
[1/5] Lendo amostra filtrada (APOGEE + Gaia): ./output_fits/apogee_gaia_sample_filtered.fits
 -> Total de objetos de entrada: 151408
 -> Caminho de fotometria configurado: /run/user/1000/gvfs/smb-share:server=150.163.28.39,share=storage-share/Samples/X-PLUS/data


FileNotFoundError: O caminho configurado não existe ou a rede caiu: /run/user/1000/gvfs/smb-share:server=150.163.28.39,share=storage-share/Samples/X-PLUS/data